<a href="https://colab.research.google.com/github/AdamClarkStandke/LLMs/blob/main/InferenceTimeScaling_BookCh4%265.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapters 4 and 5

Chapters 4 and 5 of the [book](https://www.manning.com/books/build-a-reasoning-model-from-scratch) deal with adding reasoning to LLMs through the use of *inference-time scaling *(also known as *inference-time compute scaling* or *test-time compute scaling*). *Inference-time scaling* operates during the inference stage of an LLM i.e., after the model has been trained and/or fine-tuned, and so does not require any retraining. The book focuses on three foundational inference-time techniques:
1.   **Zero-Shot Chain-of-thought (COT) Prompting**
2.   **Self-Consistency**
3.   **Self-Refinement**

**Zero-Shot Chain-of-thought (COT) Prompting** modifies a given prompt to encourage the LLM to explain its reasoning before giving an answer. **Self-Consistency** samples multiple answers generated by the LLM and then chooses the answer that appears the most. And **Self-Refinement** lets the LLM refine its answer over iterations.

Each of these methods will be implemented in the following sub-sections.** It should be noted that these methods can be combined and can be implemented even after training a reasoning LLM**.



In [1]:
# Cloning repo of Book
!git clone --depth 1 https://github.com/rasbt/reasoning-from-scratch.git

Cloning into 'reasoning-from-scratch'...
remote: Enumerating objects: 268, done.
remote: Counting objects: 100% (268/268), done.
remote: Compressing objects: 100% (245/245), done.
remote: Total 268 (delta 42), reused 137 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (268/268), 1.80 MiB | 7.00 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [2]:
# Changing directory path to import chapter code
import os
os.chdir('/content/reasoning-from-scratch/')

In [3]:
# Importing libraries
import torch
from reasoning_from_scratch.ch02 import get_device, generate_text_basic_stream_cache
from reasoning_from_scratch.ch03 import load_model_and_tokenizer, extract_final_candidate, render_prompt
from reasoning_from_scratch.qwen3 import KVCache
from collections import Counter

## Initial Setup

This sub-section details setting up the initial pretrained LLM as the base model, the prompt used to judge the reasoning of the LLM, and the wrapper function for streamed text generation.

As previoulsy detailed in the chapter two notebook, we use the [books](https://www.manning.com/books/build-a-reasoning-model-from-scratch) implementation of [Qwen3 0.6B](https://arxiv.org/pdf/2505.09388) as the pretrained base model. The prompt we create to judge the different reasoning methods is the following math question: Half the value of $3x-9$ is $x+37$. What is the value of $x$? The correct answer is 83. Lastly, the wrapper function `generate_text_stream_concat_flex` is implemented that handles the different text generation methods.

In [4]:
# Loading Qwen3-0.6B Base model
device = get_device()
model, tokenizer = load_model_and_tokenizer(
    which_model = "base",
    device = device,
    use_compile = False
)

Using CPU
qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [5]:
# Prompt used for testing
raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)
print(prompt)

You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
Half the value of $3x-9$ is $x+37$. What is the value of $x$?

Answer:


In [6]:
# Wrapper function for different inference functions
def generate_text_stream_concat_flex(
   model, tokenizer, prompt, device, max_new_tokens,
   verbose=False, generate_func=None, **generate_kwargs
):
  if generate_func is None:
    generate_func=generate_text_basic_stream_cache()

  input_ids = torch.tensor(
      tokenizer.encode(prompt),
      device=device
  ).unsqueeze(0)

  generated_ids = []
  for token in generate_func(
      model=model,
      token_ids=input_ids,
      max_new_tokens=max_new_tokens,
      eos_token_id=tokenizer.eos_token_id,
      **generate_kwargs
  ):
    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id.item())

    if verbose:
      print(
          tokenizer.decode(next_token_id.tolist()),
          end="",
          flush=True
      )
  return tokenizer.decode(generated_ids)

In [7]:
# Generated answer by non-reasoning model
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_basic_stream_cache
)

 \boxed{10}

## Zero-Shot Chain-of-thought (CoT) Prompting

This is a simple method to improve the reasoning ability of a non-reasoning LLM. This is done through the use of prompting an additional instruction that tells the model to explain its reasoning process before producing an answer. As the [book](https://www.manning.com/books/build-a-reasoning-model-from-scratch) details the simplest way to implement this is to add the phrase **Let's think step by step** or **Explain step by step** to the original prompt. This method is called **Zero-Shot Chain-of-thought (CoT) Prompting** and is implemented in the code snippet below.

CoT Prompting was first described in the paper [Chain-of-Thought Prompting Elicits Reasoning
in Large Language Models](https://arxiv.org/pdf/2201.11903) which detailed the approach of few-shot prompting/learning in which an explanation of the form of Question->COT->Answer was added to a set of prompts to illicit reasoning in an LLM. It was not until the paper of [Large Language Models are Zero-Shot Reasoners](https://arxiv.org/pdf/2205.11916) that it was shown that by just adding the phrase **Let's think step by step** to the prompt, LLMs could achieve above 50 percent performance on various reasoning benchmarks. (Note: both of these approaches can be combined to achieve even better results).






In [8]:
# Adding Let's think step by step COT to prompt
#prompt_cot_one = prompt + " \n\nLet's think step by step ."

# Adding Explain step by step COT to prompt
prompt_cot_two = prompt + " \n\nExplain step by step."
respone_cot = generate_text_stream_concat_flex(
    model, tokenizer, prompt_cot_two, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_basic_stream_cache
)

 To solve the problem, we need to find the value of \( x \) such that half the value of \( 3x - 9 \) is equal to \( x + 37 \).

### Step 1: Set up the equation
We are given that half the value of \( 3x - 9 \) is equal to \( x + 37 \). This can be written as:
\[
\frac{1}{2}(3x - 9) = x + 37
\]

### Step 2: Eliminate the fraction
To eliminate the fraction, multiply both sides of the equation by 2:
\[
2 \cdot \frac{1}{2}(3x - 9) = 2(x + 37)
\]
Simplifying both sides:
\[
3x - 9 = 2x + 74
\]

### Step 3: Solve for \( x \)
Subtract \( 2x \) from both sides to isolate \( x \):
\[
3x - 2x - 9 = 74
\]
Simplify:
\[
x - 9 = 74
\]
Add 9 to both sides to solve for \( x \):
\[
x = 74 + 9
\]
\[
x = 83
\]

### Final Answer:
\[
\boxed{83}
\]

## Self-Consistency Sampling

The next method the [book](https://www.manning.com/books/build-a-reasoning-model-from-scratch) implements is *Self-Consistency* (also known as *self-consistency sampling*) which was first detailed by the paper [Self-Consistency Improves Chain Of Thought
Reasoning In Language Models](https://arxiv.org/pdf/2203.11171). This method first generates multiple answers through the use of [Temperature Scaling](https://livebook.manning.com/wiki/categories/llm/temperature) and [Top-p filtering](https://sebastianraschka.com/faq/docs/temperature-topk-topp-sampling.html).

*Temperature* rescales the distribution of the logits produced by the LLM. This affects how the next token is generated. A *temperature* lower than 1 makes the distribution of the logits sharper while a *temperature* higher than 1 flattens the distribution of the logits. A *temperature* of 1 produces no change to the distribution and reverts back to greedy decoding i.e., argmax.

*Top-p filtering* (also known as *nucleus sampling*) is a sampling process that makes sure that low-probability tokens are not sampled. This is done by sorting the token probabilities in descending order, computing their cumulative sum, selecting tokens that satisfy the top-p cutoff, and renormalizing the remaining probabilites so that they once again sum to one. This remaining probability distribution is used for [multinomial sampling](https://docs.pytorch.org/docs/2.14/generated/torch.multinomial.html) to generate the next token. This is what produces diversity in the generated answers.

Lastly, *Self-Consistency Sampling* is done in which *Temperature scaling* and *Top-p filtering* are used to generate multiple answers to then choose the answer that is the most consistent amoung the genreated answers. The [paper](https://arxiv.org/pdf/2203.11171) used the method of majority voting to output the most consistent answer among the final answers. The function `self_consistency_vote` implements self-consistency sampling. (Note: in practice the for loop in the function `self_consistency_vote` is replaced with parallelization)

In [9]:
# Temperature scaling func
def scale_logits_by_temperature(logits, temperature):
  if temperature <= 0:
    raise ValueError("Temp must be positive")
  return logits/temperature

# Top-p filtering func
def top_p_filter(probas, top_p):
  if top_p is None or top_p >= 1.0:
    return probas

  sorted_probas, sorted_idx = torch.sort(probas, dim=1, descending=True)
  cumprobas = torch.cumsum(sorted_probas, dim=1)

  prefix = cumprobas - sorted_probas
  #print(prefix)
  keep = prefix < top_p
  #print(keep)
  keep[:, 0] = True
  #print(keep)

  kept_sorted = torch.where(
      keep, sorted_probas,
      torch.zeros_like(sorted_probas)
  )

  filtered = torch.zeros_like(probas).scatter(1, sorted_idx, kept_sorted)

  denom = torch.sum(filtered, dim=1, keepdim=True).clamp_min(1e-12)
  return filtered/denom

In [10]:
# Temp & top-p func
@torch.inference_mode()
def generate_text_temp_top_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None,
    temp=0.0,
    top_p=None
):
  model.eval()
  cache = KVCache(n_layers=model.cfg["n_layers"])
  model.reset_kv_cache()

  out = model(token_ids, cache=cache)[:, -1]
  for _ in range(max_new_tokens):

    orig_device = token_ids.device

    if temp is None or temp == 0.0:
      next_token = torch.argmax(out, dim=-1, keepdim=True)
    else:
      logits = scale_logits_by_temperature(out, temp)
      probas = torch.softmax(logits, dim=-1)
      probas = top_p_filter(probas, top_p)
      next_token = torch.multinomial(probas.cpu(), num_samples=1)
      next_token = next_token.to(orig_device)

    if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
      break

    yield next_token
    out = model(next_token, cache=cache)[:, -1]

In [11]:
# Generating reasoning text with Temp & top-p & COT
torch.manual_seed(123)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt_cot_two, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_temp_top_stream_cache,
    temp=0.5,
    top_p=0.8,
)

 To solve the problem, we need to find the value of \( x \) such that half the value of \( 3x - 9 \) is equal to \( x + 37 \). Let's break it down step by step.

### Step 1: Set up the equation
We are given that half the value of \( 3x - 9 \) is equal to \( x + 37 \). Mathematically, this can be written as:
\[
\frac{1}{2}(3x - 9) = x + 37
\]

### Step 2: Eliminate the fraction
To eliminate the fraction, multiply both sides of the equation by 2:
\[
2 \times \frac{1}{2}(3x - 9) = 2 \times (x + 37)
\]
This simplifies to:
\[
3x - 9 = 2x + 74
\]

### Step 3: Solve for \( x \)
Now, we need to isolate \( x \) on one side of the equation. Subtract \( 2x \) from both sides:
\[
3x - 2x - 9 = 74
\]
This simplifies to:
\[
x - 9 = 74
\]
Next, add 9 to both sides to solve for \( x \):
\[
x = 74 + 9
\]
\[
x = 83
\]

### Final Answer:
\[
\boxed{83}
\]

In [12]:
# Self-consistency voting func
def self_consistency_vote(
    model, tokenizer, prompt, device,
    num_samples=10, temp=0.8, top_p=0.9, max_new_tokens=2048,
    show_progress=True, show_long_answer=False, seed=None
):
  full_answers, short_answers = [], []
  for i in range(num_samples):
    if seed is not None:
      torch.manual_seed(seed + i +1)

    answer = generate_text_stream_concat_flex(
            model=model, tokenizer=tokenizer, prompt=prompt, device=device,
            max_new_tokens=max_new_tokens, verbose=show_long_answer,
            generate_func=generate_text_temp_top_stream_cache,
            temp=temp, top_p=top_p,
        )

    short = extract_final_candidate(
        answer, fallback="number_then_full"
    )

    full_answers.append(answer)
    short_answers.append(short)
    if show_progress:
      print(f"[Sample {i+1}/{num_samples}] → {short!r}")

  counts = Counter(short_answers)
  groups = {s: [] for s in counts}
  for idx, s in enumerate(short_answers):
    groups[s].append(idx)

  mc = counts.most_common()  #3
  if not mc:
    majority_winners, final_answer = [], None
  else:
    top_freq = mc[0][1]
    majority_winners = [s for s, f in mc if f == top_freq]
    final_answer = mc[0][0] if len(majority_winners) == 1 else None

  return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }

In [13]:
# Generating reasoning text using Self-consistency voting w COT
results = self_consistency_vote(
    model,
    tokenizer,
    prompt_cot_two,
    device=device,
    num_samples=5,
    temp=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    seed=123,
    show_progress=True,
)

[Sample 1/5] → '83'
[Sample 2/5] → '83'
[Sample 3/5] → '83'
[Sample 4/5] → '74'
[Sample 5/5] → '83'


In [14]:
results["final_answer"]

'83'

## Self-Refinement

As the [book](https://www.manning.com/books/build-a-reasoning-model-from-scratch) details *Self-Refinement* focuses on iteratively refining the answer produced by an LLM to potentially correct its errors. The *Self-Refinement* method relies on implementing a refinement loop in which each answer produced by the LLM is critiqued and/or scored. This critiqing and/or scoring of each answer can be done by another LLM e.g., [LLMs as Judges](https://en.wikipedia.org/wiki/LLM-as-a-Judge), or by using a scoring function and the orginial LLM to refine its initial answer. This method is what is implemented below.   

Eventhough a rule based/heuristic scorer has also been implemented in the function `heuristic_score`, the main focus of this chapter is in implementing the function `avg_logprob_answer` which is a log-probability-based scorer. This scorer is also used to train reasoning models i.e., [reinforcement learning verifible rewards](https://arxiv.org/pdf/2501.12948). *Average log-probability* (also known as *per-token log-likelihood*) calculates the log-probabilities for each answer token and then divides the sum of log-probabilities by the number of answer tokens produced. **The resulting value is a measure of the LLM's confidence in the answer it generates; however this does not mean that the answer is correct.**   

In [19]:
# Rule based Scoring
import math

def heuristic_score(
    answer,
    prompt=None,
    brevity_bonus=500.0,
    boxed_bonus=2.0,
    extract_bonus=1.0,
    fulltext_bonus=0.0
):
  score = 0.0
  cand = extract_final_candidate(answer, fallback="none")
  if cand:
    score += boxed_bonus
  else:
    cand = extract_final_candidate(answer, fallback="number_only")
    if cand:
      score += extract_bonus
    else:
      cand = extract_final_candidate(
                answer, fallback="number_then_full"
            )
      if cand:
        score += fulltext_bonus

  score += 1.5 * math.exp(-len(answer) / brevity_bonus)
  return score

In [20]:
# Avg log-prob Scoring
@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device='cpu'):
  prompt_ids = tokenizer.encode(prompt)
  answer_ids = tokenizer.encode(answer)
  full_ids = torch.tensor(prompt_ids+answer_ids, device=device)

  logits = model(full_ids.unsqueeze(0)).squeeze(0)
  logprobs = torch.log_softmax(logits, dim=-1)

  start = len(prompt_ids) - 1
  end = full_ids.shape[0] -1

  t_idx = torch.arange(start, end, device=device)
  next_tokens = full_ids[start+1:end+1]
  next_token_logp = logprobs[t_idx, next_tokens]
  return torch.mean(next_token_logp)

In [15]:
# Critique Prompt
def make_critique_prompt(raw_prompt, draft):
  return (
        "You are a meticulous reviewer. Identify logical errors, missing "
        "steps, or arithmetic mistakes. If the answer seems correct, "
        "say so briefly. Then propose a concise plan to fix issues.\n\n"
        f"Question:\n{raw_prompt}\n\n"
        f"Draft answer:\n{draft}\n\n"
        "Write a short critique and bullet-point fix plan "
        "(under ~120 words).\n"
        "Critique:"
    )

In [16]:
# Refine Prompt
def make_refine_prompt(raw_prompt, draft, critique):
  return (
        "Revise the answer using the critique. Keep it concise and "
        "end with a final boxed result: \\boxed{ANSWER}\n\n"
        f"Question:\n{raw_prompt}\n\n"
        f"Previous answer:\n{draft}\n\n"
        f"Critique:\n{critique}\n\n"
        "Revised answer:"
    )

In [22]:
def self_refinement_loop(
  model,
  tokenizer,
  raw_prompt,
  device,
  iterations=2,
  max_response_tokens=2048,
  max_critique_tokens=256,
  score_fn=None,
  prompt_renderer=render_prompt,
  prompt_suffix="",
  verbose=False,
  temperature=0.7,
  top_p=0.9
):
  steps = []

  prompt = prompt_renderer(raw_prompt) + prompt_suffix
  # Initial Answer
  current_full = generate_text_stream_concat_flex(
      model=model,
      tokenizer=tokenizer,
      prompt=prompt,
      device=device,
      max_new_tokens=max_response_tokens,
      verbose=False,
      generate_func=generate_text_temp_top_stream_cache,
      temp=temperature,
      top_p=top_p
  )
  current_extracted = extract_final_candidate(
      current_full, fallback="number_then_full"
  )
  # Initial Score
  if score_fn:
    current_score = score_fn(answer=current_full, prompt=prompt)
  else:
    current_score = 0.0

  for it in range(iterations):
    draft_before_full = current_full
    draft_before_extracted = current_extracted
    score_before = current_score

    # Critique Portion
    critique_prompt = make_critique_prompt(
        raw_prompt, draft_before_full
    )
    critique_full = generate_text_stream_concat_flex(
        model=model,
        tokenizer=tokenizer,
        prompt=critique_prompt,
        device=device,
        max_new_tokens=max_critique_tokens,
        verbose=False,
        generate_func=generate_text_temp_top_stream_cache,
        temp=temperature,
        top_p=top_p
    )

    # Refinement Portion
    refine_prompt = make_refine_prompt(
        raw_prompt, draft_before_full, critique_full
    )
    revised_full = generate_text_stream_concat_flex(
        model=model,
        tokenizer=tokenizer,
        prompt=refine_prompt,
        device=device,
        max_new_tokens=max_response_tokens,
        verbose=False,
        generate_func=generate_text_temp_top_stream_cache,
        temp=temperature,
        top_p=top_p
    )
    revised_extracted = extract_final_candidate(
        revised_full, fallback="number_then_full"
    )
    # Revised Score
    if score_fn:
      revised_score = score_fn(
          answer=revised_full, prompt=prompt
      )
    else:
      revised_score = 0.0
    # Logging Output
    step = {
        "iteration": it + 1,
        "draft_full": draft_before_full,
        "draft_extracted": draft_before_extracted,
        "critique": critique_full,
        "revised_full": revised_full,
        "revised_extracted": revised_extracted,
        "score_before": score_before,
        "score_after": revised_score,
    }
    steps.append(step)

    # Printing Output
    if verbose:
      print(
          f"[Refinement {it+1}/{iterations}]"
          f"\nCurrent: {draft_before_extracted}"
          f"\nRevised: {revised_extracted}"
          f"\nScore before: {score_before:.3f}"
          f"\nScore after: {revised_score:.3f}"
          f"\n{'=' * 25}\n"
      )

    # Revised Scoring
    if revised_score >= current_score:
      current_full = revised_full
      current_extracted = revised_extracted
      current_score = revised_score
  return {
      "final_full": current_full,
      "final_extracted": current_extracted,
      "steps": steps
  }

In [23]:
# Generating reasoning text using Self-Refinement (no scoring)
torch.manual_seed(1)

results = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)

[Refinement 1/2]
Current: 83
Revised: 83
Score before: 0.000
Score after: 0.000

[Refinement 2/2]
Current: 83
Revised: 83
Score before: 0.000
Score after: 0.000



In [24]:
# Generating reasoning text using Self-Refinement (avg_logprob scoring)
from functools import partial

avg_logprob_score = partial(
    avg_logprob_answer,
    model=model,
    tokenizer=tokenizer,
    device=device
)

torch.manual_seed(1)

results = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=avg_logprob_score,
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)

[Refinement 1/2]
Current: 83
Revised: 83
Score before: -0.070
Score after: -1.078

[Refinement 2/2]
Current: 83
Revised: 83
Score before: -0.070
Score after: -1.664



In [25]:
# Generating reasoning text using Self-Refinement (Rule scoring)
torch.manual_seed(1)

results = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=heuristic_score,
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)

[Refinement 1/2]
Current: 83
Revised: 83
Score before: 2.384
Score after: 3.376

[Refinement 2/2]
Current: 83
Revised: 83
Score before: 3.376
Score after: 3.467

